# Empirical Application: Angrist & Krueger

This notebook is a template for replicating the Angrist & Krueger (1991) instrumented study of the returns to schooling using the quarter-of-birth instruments.

The code below provides functions for OLS, TSLS, and DML-IV estimation similar to the CollegeDistance example. You must supply a suitable dataset with variables for log wages, years of education, instruments (quarters of birth), and relevant control variables.


In [ ]:
import numpy as np
import pandas as pd
import os
from sklearn.model_selection import KFold, GridSearchCV
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import statsmodels.api as sm

# Define baseline estimators (reusing functions from the CollegeDistance notebook)
def ols_beta_se(Y, D, W):
    n = len(Y)
    X = np.column_stack([D, np.ones(n), W])
    beta = np.linalg.lstsq(X, Y, rcond=None)[0]
    tau = float(beta[0])
    resid = Y - X @ beta
    sigma2 = float((resid.T @ resid)/(n - X.shape[1]))
    var = sigma2 * np.linalg.pinv(X.T @ X)
    se_tau = float(np.sqrt(var[0,0]))
    return tau, se_tau

def tsls_beta_se(Y, D, Z, W):
    n = len(Y)
    Wc = np.column_stack([np.ones(n), W])
    X = np.column_stack([D, Wc])
    Zmat = np.column_stack([Z, Wc])
    ZZ_inv = np.linalg.pinv(Zmat.T @ Zmat)
    PzY = Zmat @ (ZZ_inv @ (Zmat.T @ Y))
    PzX = Zmat @ (ZZ_inv @ (Zmat.T @ X))
    beta = np.linalg.pinv(X.T @ PzX) @ (X.T @ PzY)
    tau = float(beta[0])
    resid = Y - X @ beta
    sigma2 = float((resid.T @ resid) / (n - X.shape[1]))
    var_beta = sigma2 * np.linalg.pinv(X.T @ PzX)
    se_tau = float(np.sqrt(var_beta[0,0]))
    return tau, se_tau

# Ridge helper and PLIV functions
def fit_ridge_with_cv(X_train, y_train, inner_splits=3, seed=42):
    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('model', Ridge())
    ])
    grid = {'model__alpha': np.logspace(-2, 2, 3)}
    inner_cv = KFold(n_splits=inner_splits, shuffle=True, random_state=seed)
    gs = GridSearchCV(pipe, grid, cv=inner_cv, scoring='neg_mean_squared_error', n_jobs=1)
    gs.fit(X_train, y_train)
    return gs.best_estimator_

def pliv_tau_se(Y, D, Z, mu_hat, r_hat, pi_hat):
    n = len(Y)
    Y_tilde = Y - mu_hat
    D_tilde = D - r_hat
    Z_tilde = Z - pi_hat
    Omega = (Z_tilde.T @ Z_tilde) / n
    g_zd  = (Z_tilde.T @ D_tilde) / n
    g_zy  = (Z_tilde.T @ Y_tilde) / n
    Omega_inv = np.linalg.pinv(Omega)
    denom = float(g_zd.T @ Omega_inv @ g_zd)
    numer = float(g_zd.T @ Omega_inv @ g_zy)
    tau_hat = numer / denom
    u_hat = Y_tilde - tau_hat * D_tilde
    psi = Z_tilde * u_hat[:, None]
    Sigma = (psi.T @ psi) / n
    middle = float(g_zd.T @ Omega_inv @ Sigma @ Omega_inv @ g_zd)
    V = (1.0 / denom) * middle * (1.0 / denom)
    se = float(np.sqrt(V / n))
    ci = (tau_hat - 1.96*se, tau_hat + 1.96*se)
    return float(tau_hat), float(se), ci

def dml_iv_ridge(Y, D, Z, W, k_outer=3, k_inner=3, seed=123):
    n, dz = Z.shape
    kf = KFold(n_splits=k_outer, shuffle=True, random_state=seed)
    mu_hat = np.zeros(n)
    r_hat  = np.zeros(n)
    pi_hat = np.zeros((n, dz))
    for fold, (tr, te) in enumerate(kf.split(W)):
        W_tr, W_te = W[tr], W[te]
        mu = fit_ridge_with_cv(W_tr, Y[tr], inner_splits=k_inner, seed=seed+1000+fold)
        r  = fit_ridge_with_cv(W_tr, D[tr], inner_splits=k_inner, seed=seed+2000+fold)
        mu_hat[te] = mu.predict(W_te)
        r_hat[te]  = r.predict(W_te)
        for j in range(dz):
            pj = fit_ridge_with_cv(W_tr, Z[tr, j], inner_splits=k_inner, seed=seed+3000+37*j+fold)
            pi_hat[te, j] = pj.predict(W_te)
    return pliv_tau_se(Y, D, Z, mu_hat, r_hat, pi_hat)

# TODO: Replace the following with the actual path or URL to the Angrist & Krueger dataset.
# The dataset should include log wages, years of education, quarter-of-birth instruments, and control variables.
# Example placeholder loading code:
# url_ak = 'https://raw.githubusercontent.com/.../AngristKrueger.csv'
# ak = pd.read_csv(url_ak)
# Y = np.log(ak['wage'])
# D = ak['education']
# Z = ak[['quarter1','quarter2','quarter3']]
# W = ak[['experience','experience_squared','region', ...]]

# For now, raise an error to remind the user to supply the dataset.
raise RuntimeError('Please provide the Angrist & Krueger dataset and define Y, D, Z, and W accordingly before running the estimators.')

# Once the dataset is loaded and variables Y, D, Z, W are defined, uncomment the code below:
#
# # First-stage diagnostics
# fs = first_stage_diagnostics(D, Z, W)
# print('Angrist & Krueger first-stage diagnostics:', fs)
#
# # OLS estimate
# ols_tau, ols_se = ols_beta_se(Y, D, W)
# print('Angrist & Krueger OLS:', ols_tau, ols_se)
#
# # TSLS estimate
# tsls_tau, tsls_se = tsls_beta_se(Y, D, Z, W)
# print('Angrist & Krueger TSLS:', tsls_tau, tsls_se)
#
# # DML-IV Ridge estimate
# dml_tau, dml_se, dml_ci = dml_iv_ridge(Y, D, Z, W)
# print('Angrist & Krueger DML-IV (Ridge):', dml_tau, dml_se, dml_ci)

